# Fig. — Robustness to imperfect CSI (pilot-power sweep)

Interactive front-end for the CSI-robustness figure. Loader/summary helpers come from
`build_fig_csi.py`; **`build_figure` is inlined below as an editable cell** so you can
override `ALGORITHM_STYLE`, colors, the P_p\* marker, or which algorithms appear,
without editing the module.

- **(top)** min-SINR vs pilot power; **(bottom)** sum-SCNR vs pilot power.
- Lower pilot power → noisier MMSE channel estimates → all algorithms degrade.
  CORDIS/Centralized bend down gracefully; interference-naïve baselines cliff-edge.
- Dashed line marks the deployed operating point P_p\* (= `PILOT_POWER_DB`, 128 dB).

Unlike the other figures this one plots **all** algorithms by default (the trio
leading, benchmarks appended) — the graceful-vs-cliff contrast is the point.

**Compatible experiment:** `csi_sweep` (`kind=='sweep'`).

In [ ]:
# Make the builder + cordis importable, then apply the paper rcParams.
import sys, logging
from pathlib import Path
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

FIG_DIR = Path.cwd()
if str(FIG_DIR) not in sys.path:
    sys.path.insert(0, str(FIG_DIR))

import build_fig_csi as B   # loader + helpers (single source of truth)
from cordis.plotting import apply_paper_style, figsize, plot_sweep, save_figure
from cordis.plotting.style import ALGORITHM_STYLE   # tweak here to restyle

USE_TEX = True   # set False on a node without pdflatex
apply_paper_style()
logging.basicConfig(level=logging.INFO, format='%(levelname)-7s %(message)s')

## 1. Load the pilot-power sweep run

`RESULT_DIR = None` auto-picks the newest run (`array_<jobid>_aggregated/` preferred;
`latest` symlink never used). Pin a path to re-render a specific campaign.

In [ ]:
RESULT_DIR = None   # e.g. 'results/exp_csi_sweep/array_12345_aggregated'
result, result_dir = B.load_sweep(RESULT_DIR, experiment='csi_sweep')
print('run:', result_dir)
print('pilot grid [dB]:', sorted(result.sweep_results.keys()))

## 2. Resolve algorithms, SCNR metric, operating point

`ONLY = None` plots every algorithm present (trio first). Set a list to filter,
e.g. `['Centralized','CORDIS-ADMM','CORDIS-Split']` to drop the benchmarks.

In [ ]:
PILOT_STAR  = 128.0                  # deployed operating point [dB] (marked)
ONLY        = None                   # None -> all present, trio leading
SCNR_METRIC = None                   # None -> first available of B.SCNR_METRIC_PREFERENCE

only = B._ordered_algorithms(result, B.LEAD, only=ONLY)
scnr_metric = SCNR_METRIC or B._select_scnr_metric(result, only=only)
print('algorithms:', only)
print('scnr metric:', scnr_metric)

## 3. Numeric summary (caption sanity check)

`SINR drop` = min-SINR(best CSI) − min-SINR(worst CSI): the dynamic range across
the sweep. Compare each algorithm's value at the worst / operating / best pilot to
read off how gracefully it degrades.

In [ ]:
summary = B.collect_summary(result, only, scnr_metric, PILOT_STAR)
B._print_summary(result, summary, scnr_metric, PILOT_STAR)

## 4. `build_figure` — editable copy

The **exact** function from `build_fig_csi.py`, inlined so you can edit it here
(restyle, relabel, change the P_p\* marker, filter algorithms) and re-run. Copy it
back into the module to make an edit permanent; or mutate `ALGORITHM_STYLE` in the
setup cell to restyle without editing the body.

In [ ]:
# Bind the module-level names the function body references, so the inlined
# copy behaves identically to B.build_figure.
from typing import Optional, Sequence   # the inlined signature uses these
LEAD                    = B.LEAD
SINR_METRIC             = B.SINR_METRIC
PILOT_STAR_DEFAULT_DB   = B.PILOT_STAR_DEFAULT_DB
_SCNR_YLABEL            = B._SCNR_YLABEL
_ordered_algorithms     = B._ordered_algorithms
_select_scnr_metric     = B._select_scnr_metric
_in_range               = B._in_range
logger                  = logging.getLogger('fig_csi.nb')

In [ ]:
def build_figure(result,
                 *,
                 only: Optional[Sequence[str]] = None,
                 scnr_metric: Optional[str] = None,
                 pilot_star: float = PILOT_STAR_DEFAULT_DB,
                 log_x: bool = False,
                 use_tex: bool = True):
    """Assemble the stacked CSI-robustness figure; return ``(fig, only, scnr_metric)``.

    Built directly on ``cordis.plotting.plot_sweep`` so the paper builder stays
    decoupled from scripts/.  Reuses the project per-algorithm style.
    """
    import matplotlib
    if use_tex is False:
        matplotlib.rcParams["text.usetex"] = False
    import matplotlib.pyplot as plt
    from cordis.plotting import apply_paper_style, figsize, plot_sweep

    apply_paper_style()
    if use_tex is False:
        matplotlib.rcParams["text.usetex"] = False

    only = list(only) if only else _ordered_algorithms(result, LEAD)
    if scnr_metric is None:
        scnr_metric = _select_scnr_metric(result, only=only)

    axis = result.sweep_axis
    xlabel = axis.display or r"Pilot power $P_p/\sigma_n^2$ [dB]"

    fig, (ax_top, ax_bot) = plt.subplots(
        2, 1, figsize=figsize(width="single", aspect=3.5 / 2.6),
        sharex=True, gridspec_kw={"hspace": 0.12},
    )

    # (top) min-SINR vs pilot power
    plot_sweep(result.sweep_results, metric=SINR_METRIC, ax=ax_top,
               xlabel="", ylabel=r"min-SINR [dB]", only=only, log_x=log_x)
    ax_top.set_title("Robustness to imperfect CSI")

    # (bot) SCNR vs pilot power
    if scnr_metric is not None:
        plot_sweep(result.sweep_results, metric=scnr_metric, ax=ax_bot,
                   xlabel=xlabel,
                   ylabel=_SCNR_YLABEL.get(scnr_metric, scnr_metric),
                   only=only, log_x=log_x)
    else:
        logger.warning("No SCNR metric available; bottom panel left empty.")
        ax_bot.set_xlabel(xlabel)

    # operating-point marker P_p* (only if inside the swept range)
    if _in_range(result, pilot_star):
        for ax in (ax_top, ax_bot):
            ax.axvline(pilot_star, ls="--", lw=0.9, color="0.45", zorder=0)
        # get_xaxis_transform() = (data-x, axes-y); tight_layout-safe.
        ax_top.text(pilot_star, 0.96, r"$P_p^\star$",
                    transform=ax_top.get_xaxis_transform(),
                    ha="right", va="top", fontsize="small", color="0.35")
    else:
        logger.warning("pilot_star=%.3g dB is outside the swept range; "
                       "operating-point marker skipped.", pilot_star)

    # one legend only (top); drop the duplicate on the bottom panel
    if ax_bot.get_legend() is not None:
        ax_bot.get_legend().remove()

    # NB: no fig.tight_layout() — conflicts with the shared-x / hspace stacked
    # layout (same reason scripts/_plot_common.sweep_plot_pair omits it).
    # save_figure() uses bbox_inches='tight', which handles the margins.
    return fig, only, scnr_metric


## 5. Render

In [ ]:
fig, used_only, used_scnr_metric = build_figure(
    result, only=only, scnr_metric=scnr_metric,
    pilot_star=PILOT_STAR, use_tex=USE_TEX,
)
plt.show()

## 6. Save to `paper/figures/`

In [ ]:
out_stem = FIG_DIR.parents[1] / 'figures' / 'fig_csi'
paths = save_figure(
    fig, out_stem, formats=('pdf',),
    metadata={
        'Figure': 'fig_csi',
        'PilotStarDB': f'{PILOT_STAR:g}',
        'Algorithms': ', '.join(used_only),
        'ScnrMetric': str(used_scnr_metric),
        'Run': result_dir.name,
    },
)
for p in paths:
    print('wrote', p)